In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load the Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

# 2. Define Target, Leaks, and ID Column
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# These are the features causing the 0.9+ R2 data leakage. 
# We MUST drop them from both the training and testing sets.
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# Prepare Training Data
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# Prepare Testing Data (Save IDs for the final submission format)
test_ids = test_df[ID_COL]
X_test = test_df.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 3. Build a Preprocessing Pipeline
# Identify which columns are text/categorical and which are numbers
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Fill missing numbers with the median
numeric_transformer = SimpleImputer(strategy='median')

# Fill missing text with the most frequent value, then convert to numbers.
# handle_unknown='use_encoded_value' ensures the model doesn't crash if 
# the test set contains a category it never saw during training.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 4. Define and Train the Model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model (this might take a moment)...")
model.fit(X_train, y_train)

# 5. Predict on Test Data and Format Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

# Create the final dataframe matching the sample_submission format
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

# Save to CSV
submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'submission.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\3626102385.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model (this might take a moment)...
Predicting on test data...
Saved predictions to 'submission.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Build the Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # Feature 1: Speed Advantage (Who moves first)
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    
    # Feature 2: Level Ratio (Relative Power)
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5) # Added a tiny number to prevent divide by zero
    
    # Feature 3: Attack/Defense Matchup Ratio
    # If the move is Special, compare sp_attack to sp_defense. 
    # Otherwise, compare regular attack to defense.
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # Feature 4: Theoretical Power of the Turn
    # Combine move power, effectiveness, and stat ratios into one number
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # Feature 5: Previous HP State
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)

    danger = np.ones(len(data))
    
    # If the weather matches the opponent's type, their attacks will hit Pikachu much harder
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    
    # Hail and Sandstorm deal chip damage every turn to Pikachu, lowering expected HP
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    
    # If Electric Terrain is active, Pikachu gets a power boost, lowering the danger
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    
    data['weather_danger_level'] = danger
        
    return data

# Apply the new features to both datasets
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Model and Train
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model with new features...")
model.fit(X_train, y_train)

# 6. Predict and Create Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'engineered_submission.csv'")

Engineering features...
Training model with new features...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\1263305439.py:77: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting on test data...
Saved predictions to 'engineered_submission.csv'


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # 2.1 Stat Differentials
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2.2 Attack/Defense Ratios based on Physical vs Special moves
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # 2.3 Theoretical Power combination
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 2.4 Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 2.5 Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    return data

# Apply feature engineering
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# FILL MISSING MAX_HP to prevent NaNs during the clipping step
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Gradient Boosting Model
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        max_iter=300, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# Train
print("Training Gradient Boosting model...")
hgb_model.fit(X_train, y_train)

# 6. Predict and Post-Process (Clipping)
print("Predicting and applying physics constraints...")
raw_predictions = hgb_model.predict(X_test)

# Force predictions to be physically possible (0 to max_hp)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay9.csv', index=False)
print("Success! Saved clean predictions to 'final_hgb_submission.csv'")

Engineering features...
Training Gradient Boosting model...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\115736432.py:69: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting and applying physics constraints...
Success! Saved clean predictions to 'final_hgb_submission.csv'


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering
def engineer_features(df):
    data = df.copy()
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    return data

print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target, Remove Leaks, and Fill Missing max_hp
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define BOTH Models
print("Initializing models...")
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=150, 
        max_depth=12, 
        random_state=42, 
        n_jobs=-1
    ))
])

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        n_estimators=300, 
        learning_rate=0.05, 
        max_depth=6, 
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Train Models
print("Training Random Forest...")
rf_model.fit(X_train, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# 7. Predict and Average (The Ensemble)
print("Generating predictions...")
rf_preds = rf_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

# Give XGBoost 85% weight, and Random Forest 15% weight
ensemble_preds = (0.85 * xgb_preds) + (0.15 * rf_preds)

# Post-processing: Force physical boundaries
clipped_preds = np.clip(ensemble_preds, 0, test_max_hp)

# 8. Create Submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_preds
})

submission.to_csv('submissionDay10.csv', index=False)
print("Success! Saved clean predictions to 'submissionDay10.csv'")

Engineering features...
Initializing models...
Training Random Forest...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\2312905063.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training XGBoost...
Generating predictions...
Success! Saved clean predictions to 'ensemble_xgb_rf_submission.csv'
